In [2]:
import pandas as pd

df = pd.read_csv("http://114.207.245.181:13000/csv/ynat.csv")

In [3]:
df['predefined_news_category'].value_counts()

predefined_news_category
세계      8034
정치      7993
IT과학    7852
스포츠     7688
경제      6028
생활문화    6022
사회      2061
Name: count, dtype: int64

In [5]:
mapping = {
    "세계": 0,
    "정치": 1,
    "IT과학": 2,
    "스포츠": 3,
    "경제": 4,
    "생활문화": 5,
    "사회": 6
}

df['target'] = df['predefined_news_category'].map(mapping)

In [6]:
texts = df['title'].tolist()
labels = df['target'].tolist()

In [7]:
texts = texts[:500]
labels = labels[:500]

In [8]:
import torch
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, TensorDataset

MODEL_NAME = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
    max_length=100
)

dataset = TensorDataset(
    inputs['input_ids'],
    inputs['attention_mask'],
    torch.tensor(labels, dtype=torch.long)
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [9]:
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

class BertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)

        for param in self.bert.parameters():
            param.requires_grad = False

        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(hidden_size, 7)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask
        )

        cls = outputs['last_hidden_state'][:, 0]
        x = self.dropout(cls)
        x = self.fc1(x)

        return x

In [10]:
device  = torch.device( 
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = BertClassifier().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr = 0.001
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
epochs = 10

for epoch in range( epochs ):
    model.train()
    total_loss = 0
    
    # 추가된 부분
    correct = 0
    total = 0

    for batch in loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad() # 기울기 초기화
        logits = model( input_ids = input_ids, attention_mask = attention_mask ) # 모델 학습
        loss = criterion(logits.squeeze(), labels) # 로그 계산
        loss.backward() # 기울기 계산
        optimizer.step() # 가중치 업데이트
        total_loss += loss.item() # 전체 loss값 누적

        # 추가된 부분
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total    

    print ( f"Epoch {epoch+1} Loss { total_loss/len(loader):.4f} Accuracy {accuracy:.4f}"  )    # epoch와 loss값 출력

Epoch 1 Loss 1.2518 Accuracy 0.5920
Epoch 2 Loss 0.5837 Accuracy 0.8360
Epoch 3 Loss 0.4442 Accuracy 0.8600
Epoch 4 Loss 0.3713 Accuracy 0.8800
Epoch 5 Loss 0.2783 Accuracy 0.9320
Epoch 6 Loss 0.2469 Accuracy 0.9260
Epoch 7 Loss 0.2236 Accuracy 0.9400
Epoch 8 Loss 0.1812 Accuracy 0.9520
Epoch 9 Loss 0.1821 Accuracy 0.9480
Epoch 10 Loss 0.1606 Accuracy 0.9520


In [47]:
# mapping의 키 값을 바꾸기
id_to_label = {v: k for k, v in mapping.items()}

print(id_to_label)


def predict_news(text):

    model.eval()

    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=100,
        return_tensors="pt"
    )

    input_ids = ( enc["input_ids"].to(device) )
    attention_mask = ( enc["attention_mask"].to(device) )


    with torch.no_grad():
        logits = model(
            input_ids,
            attention_mask
        )

        pred = torch.argmax(
            logits,
            dim=1
        ).item()

    return id_to_label[pred]


print( predict_news( "삼성전자가 새로운 AI 반도체를 공개했다") )
print( predict_news( "손흥민이 결승골을 터뜨렸다") )
print( predict_news( "국회가 예산안을 통과시켰다") )
print( predict_news(""))

{0: '세계', 1: '정치', 2: 'IT과학', 3: '스포츠', 4: '경제', 5: '생활문화', 6: '사회'}
IT과학
스포츠
정치
경제
